In [1]:
import pandas as pd
from rdkit.Chem import MolFromSmiles
import torch
import sys
sys.path.append('../')
import FragShapley
from torch_geometric.utils import from_rdmol
import os

from torch_geometric.explain import Explainer, GNNExplainer, CaptumExplainer, PGExplainer

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# all examples done with esol delaney dataset for easier comparison

In [3]:
fig_folder = "figures/"

# GCN with GNNExplainer

In [4]:
# load dataset and select split
split = 0
model_type = "GCN"
task = 'regression'
df_expl = pd.read_pickle(f"../3_solubility/{model_type.lower()}_regression_solubility/df_explanation.pkl")
dataset = 'esol_delaney'

df_expl = df_expl.query("dataset == @dataset")
df_expl['n_fragments'] = df_expl.fragExplainer_result.apply(len)

In [5]:
# Load Model
model = torch.load(f'../3_solubility/{model_type.lower()}_regression_solubility/models/model_{model_type.lower()}r_esol_delaney_split_{split}.pkl', weights_only=False)

In [ ]:
# select example to explain
df_tmp = df_expl.query("n_fragments >= 3 and split == @split")
rows = [12, 16, 20, 23, 26, 39]

for row_id in rows:

    # extract data
    sample_row = df_tmp.iloc[row_id]
    smiles = sample_row.smiles
    mol = MolFromSmiles(smiles)

    # Load Example in Data
    data = from_rdmol(mol)
    data

    # try out explainer
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=200),
        explanation_type='model',
        node_mask_type='object', # explain important nodes
        model_config=dict(
            mode='regression',
            task_level='graph',
            return_type='raw',
        ),
    )
    explanation = explainer(data.x, data.edge_index)
    print(f'GNNE: MIN:{min(explanation.node_mask)}, MAX: {max(explanation.node_mask)}')
    
    out_GNNE = FragShapley.visualization.visualize_atom_contributions_from_mol_new(mol,
                                                                                   atom_contributions=[float(i) for i in explanation.node_mask], 
                                                                                   min=0.12,
                                                                                   max=0.19,
                                                                                   colormap="Wistia",
                                                                                   rad=0.3,
                                                                                   legend='')
    with open(os.path.join(fig_folder, f"molecules_comparison/5_solubility_GCN_GNNExplainer_row_{row_id}.svg"), "w") as f:
        f.write(out_GNNE.data)
    

    # now visualize the FragShapley approach
    print(f'fE: MIN:{min(sample_row.fragExplainer_result.values())}, MAX: {max(sample_row.fragExplainer_result.values())}')
    contributions_fE = FragShapley.visualization.get_atom_contribution_from_result_dict(smiles,
                                                                                        results_dict=sample_row.fragExplainer_result,
                                                                                        frag_to_atom_ids=sample_row.frag_to_atom_ids)
    out_fE = FragShapley.visualization.visualize_contributions_new(smiles=smiles,
                                                                   contributions=sample_row.fragExplainer_result.values(),
                                                                   frag_to_atom_ids=sample_row.frag_to_atom_ids,
                                                                   min=-5,
                                                                   max=5,
                                                                   colormap='bwr')
    with open(os.path.join(fig_folder, f"molecules_comparison/5_solubility_GCN_fragExplainer_row_{row_id}.svg"), "w") as f:
        f.write(out_fE.data)


GNNE: MIN:tensor([0.1155]), MAX: tensor([0.1550])
fE: MIN:-3.2186924417813616, MAX: 0.7934256196022031
GNNE: MIN:tensor([0.1349]), MAX: tensor([0.1873])
fE: MIN:-2.1981983900070188, MAX: 0.8347640911738078
GNNE: MIN:tensor([0.1252]), MAX: tensor([0.1666])
fE: MIN:-2.292810487747192, MAX: 1.1883131305376688
GNNE: MIN:tensor([0.1267]), MAX: tensor([0.1521])
fE: MIN:-1.3434414466222127, MAX: 0.44332615534464515
GNNE: MIN:tensor([0.1223]), MAX: tensor([0.1604])
fE: MIN:-2.886659387747447, MAX: 1.4611727878451344
GNNE: MIN:tensor([0.1246]), MAX: tensor([0.1638])
fE: MIN:-1.6730170261292234, MAX: 1.2511146728481564
